# Genesis Environment — Training Notebook

**6 cells. Three things:**
1. Setup
2. Self-improvement loop (tool evolution) + visualisation
3. GRPO training (agent LLM) + combined loop + visualisation

Runs on a T4 (Google Colab free tier). Set your tokens in the `.env` cell below.

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import os, subprocess

# Clone repo
if not os.path.exists("genesis_env"):
    subprocess.run(["git", "clone", "https://huggingface.co/spaces/berlin1906/genesis_env"], check=True)
os.chdir("genesis_env")

# Install dependencies
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "trl>=1.2.0", "peft>=0.10", "bitsandbytes", "accelerate", "datasets"], check=True)

# Set env tokens — fill these in
os.environ["HF_TOKEN"]    = "hf_..."   # HuggingFace token
os.environ["MODEL_NAME"]  = "Qwen/Qwen2.5-Coder-1.5B-Instruct"  # or 7B if you have the VRAM
os.environ["RUBRIC_MODEL"] = "Qwen/Qwen2.5-Coder-7B-Instruct"
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # optional fallback for reasoning scorer

print("Setup complete.")

In [ ]:
# ── Cell 2: Self-Improvement Loop (Tool Evolution) ─────────────────────────
#
# Each cycle:
#   ① Run N episodes → grade every tool call
#   ② EMA tracker flags the worst tool
#   ③ Tool Architect rewrites it
#   ④ Re-evaluate → measure delta
#   ⑤ delta > 0 → keep  |  delta ≤ 0 → revert

import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

from training.self_improve import run_loop

CYCLES    = 3   # number of tool-evolution cycles
EPISODES  = 3   # evaluation episodes per cycle

run_loop(n_cycles=CYCLES, n_episodes=EPISODES, dry_run=False)

# Load persisted history for visualisation
state_path = Path("training/loop_state.json")
si_history = json.loads(state_path.read_text())["history"] if state_path.exists() else []
print(f"\nSelf-improve history ({len(si_history)} cycles loaded)")

In [ ]:
# ── Cell 3: Self-Improvement Visualisation ─────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if not si_history:
    print("No self-improve history to plot. Run Cell 2 first.")
else:
    cycles      = [h["cycle"]        for h in si_history]
    before_rew  = [h["before_reward"] for h in si_history]
    after_rew   = [h["after_reward"]  for h in si_history]
    deltas      = [h["delta"]         for h in si_history]
    reverted    = [h.get("reverted", False) for h in si_history]
    tools       = [h.get("target_tool", "")  for h in si_history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Self-Improvement Loop — Tool Evolution", fontsize=14, fontweight="bold")

    # ── Left: before vs after reward per cycle
    x = np.arange(len(cycles))
    w = 0.35
    axes[0].bar(x - w/2, before_rew, w, label="Before rewrite", color="#7986CB", alpha=0.85)
    axes[0].bar(x + w/2, after_rew,  w, label="After rewrite",  color="#4DB6AC", alpha=0.85)
    for i, rev in enumerate(reverted):
        if rev:
            axes[0].annotate("↩ reverted", xy=(x[i] + w/2, after_rew[i] + 0.02),
                             ha="center", fontsize=8, color="#E53935")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"C{c}\n{t}" for c, t in zip(cycles, tools)], fontsize=8)
    axes[0].set_ylabel("Mean Episode Reward")
    axes[0].set_ylim(0, 1.1)
    axes[0].set_title("Reward Before vs After Tool Rewrite")
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    # ── Right: delta per cycle (green = kept, red = reverted)
    bar_colors = ["#E53935" if r else "#43A047" for r in reverted]
    axes[1].bar(x, deltas, color=bar_colors, alpha=0.85)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f"C{c}" for c in cycles])
    axes[1].set_ylabel("Reward Delta")
    axes[1].set_title("Improvement Delta per Cycle")
    kept_patch    = mpatches.Patch(color="#43A047", label="Kept")
    reverted_patch = mpatches.Patch(color="#E53935", label="Reverted")
    axes[1].legend(handles=[kept_patch, reverted_patch])
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("self_improve_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → self_improve_results.png")

In [ ]:
# ── Cell 4: GRPO Training (Agent LLM only) ─────────────────────────────────
#
# Fine-tunes the agent LLM with GRPO using TRL + QLoRA.
# Reward signal = composite episode score from GenesisEnvironment.
# Runs GRPO only — no tool evolution.

from training.combined_loop import run_combined_loop

GRPO_CYCLES     = 2
GRPO_BATCH_SIZE = 4   # tasks per GRPO update

run_combined_loop(
    n_cycles=GRPO_CYCLES,
    n_episodes=2,
    batch_size=GRPO_BATCH_SIZE,
    grpo_only=True,       # tool evolution disabled
    improve_only=False,
    dry_run=False,
    use_local_model=False,
)

print("\nGRPO training complete. Checkpoints saved to checkpoints/grpo/")

In [ ]:
# ── Cell 5: Combined Loop (GRPO + Tool Evolution) ──────────────────────────
#
# Each cycle runs both phases:
#   Phase 1 — GRPO: fine-tune agent LLM weights on a batch of tasks
#   Phase 2 — Self-improve: evaluate → Tool Architect rewrites worst tool → keep/revert
#
# Both improve in lockstep every cycle.

import json
from pathlib import Path

COMBINED_CYCLES    = 3
COMBINED_EPISODES  = 3
COMBINED_BATCH     = 4

run_combined_loop(
    n_cycles=COMBINED_CYCLES,
    n_episodes=COMBINED_EPISODES,
    batch_size=COMBINED_BATCH,
    grpo_only=False,
    improve_only=False,
    dry_run=False,
    use_local_model=True,   # use the GRPO-updated model for self-improve inference
)

# Load combined history for visualisation
combined_path = Path("training/combined_state.json")
combined_history = json.loads(combined_path.read_text())["history"] if combined_path.exists() else []
print(f"\nCombined loop history ({len(combined_history)} cycles loaded)")

In [ ]:
# ── Cell 6: Combined Loop Visualisation ────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

if not combined_history:
    print("No combined history to plot. Run Cell 5 first.")
else:
    cycles      = [h["cycle"]                     for h in combined_history]
    grpo_loss   = [h.get("grpo_mean_loss")         for h in combined_history]
    grpo_rew    = [h.get("grpo_mean_reward")        for h in combined_history]
    before_rew  = [h.get("before_reward")           for h in combined_history]
    after_rew   = [h.get("after_reward")            for h in combined_history]
    deltas      = [h.get("delta", 0)               for h in combined_history]
    reverted    = [h.get("reverted", False)         for h in combined_history]
    tools       = [h.get("target_tool", "--")       for h in combined_history]

    x = np.arange(len(cycles))
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Combined Loop — GRPO + Tool Evolution", fontsize=15, fontweight="bold")

    # ── Top-left: GRPO loss over cycles
    ax = axes[0, 0]
    valid_loss = [(i, v) for i, v in enumerate(grpo_loss) if v is not None]
    if valid_loss:
        xi, yi = zip(*valid_loss)
        ax.plot(xi, yi, marker="o", color="#7986CB", linewidth=2)
        ax.fill_between(xi, yi, alpha=0.15, color="#7986CB")
    ax.set_title("GRPO Training Loss")
    ax.set_xlabel("Cycle")
    ax.set_ylabel("Mean Loss")
    ax.set_xticks(x)
    ax.grid(alpha=0.3)

    # ── Top-right: GRPO reward over cycles
    ax = axes[0, 1]
    valid_rew = [(i, v) for i, v in enumerate(grpo_rew) if v is not None]
    if valid_rew:
        xi, yi = zip(*valid_rew)
        ax.plot(xi, yi, marker="o", color="#4DB6AC", linewidth=2)
        ax.fill_between(xi, yi, alpha=0.15, color="#4DB6AC")
    ax.set_title("GRPO Mean Reward")
    ax.set_xlabel("Cycle")
    ax.set_ylabel("Mean Reward")
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.grid(alpha=0.3)

    # ── Bottom-left: before vs after tool rewrite
    ax = axes[1, 0]
    w = 0.35
    valid_before = [v if v is not None else 0 for v in before_rew]
    valid_after  = [v if v is not None else 0 for v in after_rew]
    ax.bar(x - w/2, valid_before, w, label="Before rewrite", color="#7986CB", alpha=0.85)
    ax.bar(x + w/2, valid_after,  w, label="After rewrite",  color="#4DB6AC", alpha=0.85)
    for i, rev in enumerate(reverted):
        if rev:
            ax.annotate("↩", xy=(x[i] + w/2, valid_after[i] + 0.02),
                        ha="center", fontsize=11, color="#E53935")
    ax.set_xticks(x)
    ax.set_xticklabels([f"C{c}\n{t}" for c, t in zip(cycles, tools)], fontsize=8)
    ax.set_ylabel("Mean Episode Reward")
    ax.set_ylim(0, 1.1)
    ax.set_title("Tool Rewrite: Before vs After")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # ── Bottom-right: delta per cycle
    ax = axes[1, 1]
    bar_colors = ["#E53935" if r else "#43A047" for r in reverted]
    ax.bar(x, deltas, color=bar_colors, alpha=0.85)
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels([f"C{c}" for c in cycles])
    ax.set_ylabel("Reward Delta")
    ax.set_title("Tool Evolution Delta per Cycle")
    kept_patch     = mpatches.Patch(color="#43A047", label="Kept")
    reverted_patch = mpatches.Patch(color="#E53935", label="Reverted")
    ax.legend(handles=[kept_patch, reverted_patch])
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig("combined_loop_results.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → combined_loop_results.png")